# Sentinel-2 L2A: mosaicos y clasificación de fuego

Prepare Colab → cargue vectores → inspeccione GEE → descargue y compare → dibuje muestras → clasifique con RF.

Edite los parámetros aquí; la lógica está en `Sentinel2_pipeline_test.py`.
Las escenas se procesan en `/content/sentinel2_temporal`; el mosaico terminado se conserva en Drive.

## 1. Configuración y preparación de Colab

Edite las variables de la primera celda y ejecute ambas celdas en orden.
La segunda celda reemplaza el clon en `/content/fire_monitor`, instala dependencias y configura la autenticación de GEE/GCS mediante `M0_auth_config.py`.
Los títulos de las celdas permiten ocultar el código; los resultados se guardan fuera del clon.


In [ ]:
#@title 1. Variables del proyecto
COUNTRY_SELECTED = "colombia"
CAMPAIGN_SELECTED = "MONITOR_01"
MOSAIC_YEARS = [2025, 2026]

REPO_URL = "https://github.com/mapbiomas/colombia-fire.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/fire_monitor"
CORE_RELATIVE_PATH = "mapbiomas_fire_monitor/version_01/src/core"
DIRECTORIO_TRABAJO = "/content/sentinel2_temporal"

# Proyecto y regiones utilizados actualmente por este notebook.
GEE_PROJECT = "mapbiomas-fire-485203"
ASSET_REGIONS = "projects/mapbiomas-colombia/assets/FIRE/AUXILIARY_DATA/regiones_monitor_fuego"
ASSET_TILES = "projects/mapbiomas-colombia/assets/DATOS_AUXILIARES/VECTORES/SENTINEL/sentinel_2_index_shapefile"

GLOBAL_CONFIG = {
    "country": COUNTRY_SELECTED,
    "campaign": CAMPAIGN_SELECTED,
    "sensor": "sentinel2",
    "periodicity": "monthly",
    "mosaic_methods": ["minnbr", "minnbr_buffer"],
    "personal_task_flag": "MONITOR",
    "language": "en",  # en | es | pt | fr | id
    "gcs_bucket": "mapbiomas-fire",
    "gcs_library_images_prefix": f"sudamerica/{COUNTRY_SELECTED}/CATALOG_01",
    "gcs_campaigns_prefix": f"sudamerica/{COUNTRY_SELECTED}/CATALOG_01",
    "gee_project": GEE_PROJECT,
    "gee_library_images_prefix": f"projects/mapbiomas-{COUNTRY_SELECTED}/assets/FIRE/CATALOG_01",
    "gee_campaigns_prefix": f"projects/mapbiomas-{COUNTRY_SELECTED}/assets/FIRE/CATALOG_01",
    "asset_regions": ASSET_REGIONS,
}


In [ ]:
#@title 2. Preparar Colab y autenticar
import os
import shutil
import subprocess
import sys
from pathlib import Path
from importlib import import_module, invalidate_caches, reload

# Crear siempre un clon nuevo, incluso al repetir desde otro directorio.
os.chdir("/content")
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Clon anterior eliminado.")
subprocess.run(
    ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
    check=True,
)

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "gdal-bin", "python3-gdal"], check=True)
%pip install -q gdown "geopandas>=1.1" "pystac-client>=0.9" "rasterio>=1.4" "folium>=0.20" matplotlib tqdm planetary-computer earthengine-api geemap scikit-learn scipy gcsfs psutil google-api-python-client "ipywidgets>=8,<9" "jupyter-loopback[comm]>=0.3,<1"

CARPETA_MODULO = Path(REPO_DIR) / CORE_RELATIVE_PATH
MODULOS_REQUERIDOS = (
    "M0_auth_config", "M_mosaics", "M_lang", "M_gcs", "M_cache", "Sentinel2_pipeline_test",
    "M1_export_logic", "M_blocklist", "M1_mosaic_preview_ui",
)
faltantes = [f"{name}.py" for name in MODULOS_REQUERIDOS if not (CARPETA_MODULO / f"{name}.py").is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan archivos en {CARPETA_MODULO}: {', '.join(faltantes)}")
sys.path = [p for p in sys.path if Path(p or os.curdir).resolve() != CARPETA_MODULO.resolve()]
sys.path.insert(0, str(CARPETA_MODULO))
os.chdir(CARPETA_MODULO)
invalidate_caches()

# Recargar la configuración y sus dependencias después de renovar el clon.
for name in MODULOS_REQUERIDOS:
    sys.modules.pop(name, None)
from M0_auth_config import set_global_opts, authenticate, print_config

set_global_opts(**GLOBAL_CONFIG)
authenticate()
print_config()

NotebookWorkflow = reload(import_module("Sentinel2_pipeline_test")).NotebookWorkflow
workflow = NotebookWorkflow(Path(DIRECTORIO_TRABAJO))


## Mosaico mensual: vista y exportación GEE / GCS

Elija año, mes completo y método; pulse **Preparar y visualizar**. La vista y las exportaciones utilizan el mismo mosaico de `M1_export_logic`: Cloud Score Plus ≥ 0.40, mínimo NBR y, para `minnbr_buffer`, la máscara INPE cuando está disponible.

La cobertura es toda Colombia según `ASSET_REGIONS`. La salida conserva la escala de M1: bandas espectrales en byte (reflectancia × 100), día del año en int16 y exportación a 10 m. Los parámetros SCL y nubosidad de la vista regional de abajo pertenecen al flujo de descarga local.

Seleccione bandas y destinos y pulse **Exportar mosaico**; se inicia una tarea por banda y destino. GEE usa las colecciones de `GLOBAL_CONFIG` y GCS guarda GeoTIFF COG en la carpeta `CHUNKS`. Pulse **Actualizar tareas** para consultar su estado.


In [ ]:
#@title Mosaico mensual y tareas de exportación
from M1_mosaic_preview_ui import run_ui, start_export

ui_exporter = run_ui(years=MOSAIC_YEARS)

# Alternativa al botón Exportar mosaico, después de preparar la vista:
# start_export(ui_exporter)


## 2. Shapefiles de Drive

Los IDs incluyen el índice de teselas y las 26 regiones de monitoreo.
Active `AUTENTICAR_DRIVE` si los archivos son privados.

In [ ]:
TILES_ID = "1sJlyT9TfGx2Qb6gW4ptupKsPnc-oa39d"
REGIONS_ID = "16s0_U649udFVTXGRwHHjsHbHFv3WKD98"
AUTENTICAR_DRIVE = True  #@param {type:"boolean"}

workflow.load_vectors(TILES_ID, REGIONS_ID, authenticate=AUTENTICAR_DRIVE)

DRIVE_ROOT = "/content/drive/MyDrive/FuegoMonitor"
workflow.use_drive(DRIVE_ROOT)


## Inspeccionar en GEE

Edite `PARAMETERS` y autentique su cuenta. La fecha final es **exclusiva**.
Las bandas y la máscara SCL se comparten con la descarga.

GEE muestra el mosaico según el zoom; `resolution_m` controla la salida local.
Active `MOSTRAR_ESTADISTICAS_GEE` para consultar conteos y fechas.

In [ ]:
# Lista única de parámetros: GEE y descarga usan estos mismos valores.
PARAMETERS = {
    "project": GLOBAL_CONFIG["gee_project"],
    "regions_asset": GLOBAL_CONFIG["asset_regions"],
    "tiles_asset": ASSET_TILES,
    "region_id": 7,
    "start_date": "2026-01-01",
    "end_date": "2026-02-01",  # exclusiva: incluye todo enero, no el 1 de febrero
    "max_cloud_cover": 20,
    "bands": ["B11", "B8", "B4", "B12"],  # amplíe esta lista para descargar más bandas
    "resolution_m": 20,  # resolución del mosaico descargado
    "scl_exclude": [0, 1, 2, 3, 8, 9, 10, 11],
    "vis": {"bands": ["swir1", "nir", "red"], "min": 0.03, "max": 0.4},
}
# Rojo, NIR, SWIR1, SWIR2 se incluyen siempre para los índices; SCL es auxiliar.


In [ ]:
MOSTRAR_ESTADISTICAS_GEE = True  #@param {type:"boolean"}

Map = workflow.preview(PARAMETERS, include_metadata=MOSTRAR_ESTADISTICAS_GEE)
Map


## 5. Descargar y componer

Active `DESCARGAR_IMAGENES`. Se reutiliza el mosaico guardado para las mismas fechas, región, fuente y parámetros.
`SOBRESCRIBIR_MOSAICO = True` fuerza una nueva descarga y composición.

Se procesa temporalmente y se guarda el COG, metadatos y tiempos en `DRIVE_ROOT/mosaics/AAAA-MM`.
`CONSERVAR_IMAGENES = False` elimina las escenas tras procesarlas.
Los tiempos de descarga y procesamiento se solapan.


In [ ]:
SOURCE = "planetary-computer"  #@param ["planetary-computer", "earth-search"]
DESCARGAR_IMAGENES = True  #@param {type:"boolean"}
CONSERVAR_IMAGENES = False  #@param {type:"boolean"}

SOBRESCRIBIR_MOSAICO = False  #@param {type:"boolean"}

composicion = workflow.download(
    PARAMETERS, source=SOURCE, enabled=DESCARGAR_IMAGENES,
    keep_inputs=CONSERVAR_IMAGENES, overwrite=SOBRESCRIBIR_MOSAICO,
)


## 6. Comparación pausada

La comparación está comentada. Continúe en la sección 7 para abrir el mosaico y tomar muestras.


In [ ]:
# VIEW = "color"  #@param ["color", "NBR"]
# RENOVAR_TESELAS_GEE = False  #@param {type:"boolean"}

# comparison = workflow.compare(
#     PARAMETERS, source=SOURCE, view=VIEW, refresh_gee=RENOVAR_TESELAS_GEE,
# )
# comparison

## 7. Muestras, modelos y clasificación

Edite muestras en Folium y elija el archivo para entrenar.
Seleccione **Crear modelo nuevo** y el algoritmo, o aplique/reentrene uno guardado.
Los resultados quedan como capas independientes; el panel muestra tiempo y RAM.


In [ ]:
MODEL_PARAMETERS = {
    "gaussian_nb": {"var_smoothing": 1e-9},
    "logistic_regression": {"C": 1.0, "max_iter": 1000},
    "svm": {"C": 1.0, "gamma": "scale"},
    "extra_trees": {},
    "gradient_boosting": {"learning_rate": 0.1, "max_depth": 3},
}
RF_PARAMETERS = {
    "classifier_parameters": MODEL_PARAMETERS,
    "n_estimators": 200, "max_depth": 20,
    "min_samples_leaf": 2,
    "max_samples_per_class": 5000, "min_samples_per_class": 10,
    "random_state": 42, "n_jobs": 2, "block_size": 512,
}


In [ ]:
MOSAIC_PATH = ""  # vacío: recuperar el mosaico de PARAMETERS
SOURCE = globals().get("SOURCE", "planetary-computer")

training = workflow.training_canvas(
    PARAMETERS, source=SOURCE, mosaic_path=MOSAIC_PATH or None, rf_parameters=RF_PARAMETERS,
)
training.show()


## Fuentes Sentinel-2 L2A

Reflectancia de superficie ortorrectificada y corregida atmosféricamente.
Cuotas consultadas el **19 de octubre de 2026**.

| Fuente | Cuota gratuita | Requisitos |
| --- | --- | --- |
| [Earth Search / AWS — COG de Nivel 2A](https://registry.opendata.aws/sentinel-2-l2a-cogs/) **(opción de descarga)** | Acceso gratuito; la documentación no especifica una cuota mensual en GB. | Ninguno para descargar COG públicos por HTTPS. |
| [Copernicus Data Space — productos de Nivel 2A](https://documentation.dataspace.copernicus.eu/Data/SentinelMissions/Sentinel2.html#sentinel-2-level-2a-surface-reflectance) | [12 TB por cada período móvil de 30 días a velocidad normal; después se reduce la velocidad.](https://documentation.dataspace.copernicus.eu/Quotas.html) | Cuenta gratuita de CDSE; token de acceso o credenciales S3. |
| [CDSE Sentinel Hub — API de procesamiento L2A](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Data/S2L2A.html) | [10.000 unidades de procesamiento y 10.000 solicitudes al mes.](https://documentation.dataspace.copernicus.eu/Quotas.html) | Cuenta CDSE y cliente OAuth; solicitar bandas espectrales en unidades REFLECTANCE. |
| [Microsoft Planetary Computer — sentinel-2-l2a](https://planetarycomputer.microsoft.com/dataset/sentinel-2-l2a) | [Acceso gratuito; no se especifica una cuota mensual en GB; hay límites de frecuencia al solicitar tokens.](https://planetarycomputer.microsoft.com/docs/concepts/sas/) | URL firmadas con SAS; se pueden obtener tokens de forma anónima. |
| [Google Earth Engine — S2_SR_HARMONIZED](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED) | [Nivel Comunidad: 150 horas-EECU por proyecto y mes; rendimiento reducido al superar la cuota.](https://developers.google.com/earth-engine/guides/noncommercial_tiers) | Cuenta Google, proyecto de Cloud registrado, uso no comercial verificado y autenticación. |

Descarga: Earth Search o Planetary Computer. Inspección: GEE.
Las cuotas de descarga y cómputo son distintas; una cuota no publicada no implica acceso ilimitado.